# Embeddings

Loads the same preprocessed GoEmotions dataset as Project 1 (via `projects/src/data.py`).

In [5]:
# Setup: make the course code (part1) and our shared code (projects/src) importable.
# Run from projects/project2 (VS Code / JupyterLab do this by default).
import sys
from pathlib import Path

PROJECT2_DIR = Path.cwd().resolve()
REPO_ROOT = PROJECT2_DIR.parents[1]
PROJECTS_DIR = REPO_ROOT / "projects"
for p in (REPO_ROOT, PROJECTS_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import pandas as pd

from src import config, data

## Load preprocessed dataset

In [7]:
LABEL_MODE = "fine"   # "fine" (28) | "ekman" (7) | "sentiment" (4)
pairs, outcome_space, stats = data.load_pairs(label_mode=LABEL_MODE)
print(stats)

df = pd.DataFrame(pairs, columns=["text", "label"])
df["label_name"] = df["label"].map(outcome_space.name)
print(len(df), outcome_space)
df.head()

{'label_mode': 'fine', 'min_raters': 2, 'comments_with_label': 54263, 'kept': 45446, 'dropped_multilabel': 8817}
45446 0: admiration, 1: amusement, 2: anger, 3: annoyance, 4: approval, 5: caring, 6: confusion, 7: curiosity, 8: desire, 9: disappointment, 10: disapproval, 11: disgust, 12: embarrassment, 13: excitement, 14: fear, 15: gratitude, 16: grief, 17: joy, 18: love, 19: nervousness, 20: optimism, 21: pride, 22: realization, 23: relief, 24: remorse, 25: sadness, 26: surprise, 27: neutral


,text,label,label_name
0,Fast as [NAME] will carry me. Seriously uptown...,4,approval
1,You blew it. They played you like a fiddle.,27,neutral
2,TL;DR No more Superbowls for [NAME]. Get ready...,9,disappointment
3,So much time saved. Not.,7,curiosity
4,Emotes have a ridiculous amount of effort put ...,27,neutral


## Pretrained embeddings: GloVe Twitter (27B tokens, 1.2M vocab)

The downloaded zip holds four files (25d / 50d / 100d / 200d), but `part2.clslib.PretrainedEmbeddings` only accepts a zip with exactly one file. So we extract the size we want once into `data/cache/` (git-ignored) and load the `.txt` from there.

In [8]:
import zipfile

import part2.clslib

GLOVE_ZIP = PROJECTS_DIR / "src" / "embedders" / "glove.twitter.27B.zip"

  # GloVe Twitter zip with all 4 sizes
CACHE_DIR = PROJECT2_DIR / "data" / "cache"                  # git-ignored


def download_pretrained(dims=200):
    """
    Like the tutorial's download_pretrained, but for GloVe Twitter.
    Extracts glove.twitter.27B.{dims}d.txt from GLOVE_ZIP (skipped if already there).
    Returns (path, dims, skip_first_line) for PretrainedEmbeddings.
    """
    if dims not in (25, 50, 100, 200):
        raise ValueError(f"GloVe Twitter has 25/50/100/200 dims, not {dims}")
    name = f"glove.twitter.27B.{dims}d.txt"
    destination = CACHE_DIR / name
    if not destination.is_file():
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(GLOVE_ZIP) as archive:
            archive.extract(name, CACHE_DIR)
    return str(destination), dims, False   # GloVe files have no header line

In [9]:
%%time
embtable_version, embtable_dims, embtable_metadata = download_pretrained(200)
embtable_version, embtable_dims, embtable_metadata

CPU times: user 5.06 s, sys: 604 ms, total: 5.67 s
Wall time: 5.69 s


('/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/projects/project2/data/cache/glove.twitter.27B.200d.txt',
 200,
 False)

In [10]:
%%time
# a few minutes for the 200d file (1.2M words)
trained_emb_model = part2.clslib.PretrainedEmbeddings(embtable_version, embtable_dims, skip_first_line=embtable_metadata)

CPU times: user 46.7 s, sys: 1.04 s, total: 47.8 s
Wall time: 47.8 s


In [11]:
print("Shape of embedding matrix:", trained_emb_model.vectors.shape)
print("Size of vocabulary:", len(trained_emb_model.vocab))
print(f"UNK token {trained_emb_model.vocab.default_index}:{trained_emb_model.vocab.default_word}")
print(f"PAD token {trained_emb_model.vocab.index(trained_emb_model.pad_token)}:{trained_emb_model.pad_token}")
print("First few entries in the vocabulary:")
for _, (i, w) in zip(range(10), trained_emb_model.vocab.items()):
    print(f"{i}:{w}")

Shape of embedding matrix: (1193515, 200)
Size of vocabulary: 1193515
UNK token 1:<unk>
PAD token 0:<pad>
First few entries in the vocabulary:
0:<pad>
1:<unk>
2:<user>
3:.
4::
5:rt
6:,
7:<repeat>
8:<hashtag>
9:<number>


## MLP classifier (embed-and-pool → MLP → log-softmax)

### Tokenizer

In [12]:
from collections import Counter
from part2 import clslib
import nltk

nltk.download("punkt_tab", quiet=True)

tokenizer = clslib.BasicTokenizer(
    lowercase=True,
    remove_punct=False,
    remove_stopwords=False,
)

# Check how the tokenizer handles Reddit comments
for text, label in pairs[:5]:
    print("Original:", text)
    print("Tokens:  ", tokenizer.tokens(text))
    print()

Original: Fast as [NAME] will carry me. Seriously uptown to downtown via Claiborne or broad isn’t that bad
Tokens:   ['fast', 'as', '[', 'name', ']', 'will', 'carry', 'me', '.', 'seriously', 'uptown', 'to', 'downtown', 'via', 'claiborne', 'or', 'broad', 'isn', '’', 't', 'that', 'bad']

Original: You blew it. They played you like a fiddle.
Tokens:   ['you', 'blew', 'it', '.', 'they', 'played', 'you', 'like', 'a', 'fiddle', '.']

Original: TL;DR No more Superbowls for [NAME]. Get ready for another winning season that ends in disappointment.
Tokens:   ['tl', ';', 'dr', 'no', 'more', 'superbowls', 'for', '[', 'name', ']', '.', 'get', 'ready', 'for', 'another', 'winning', 'season', 'that', 'ends', 'in', 'disappointment', '.']

Original: So much time saved. Not.
Tokens:   ['so', 'much', 'time', 'saved', '.', 'not', '.']

Original: Emotes have a ridiculous amount of effort put into them, which only makes the fact that nobody uses them sadder
Tokens:   ['emotes', 'have', 'a', 'ridiculous', 'am

In [13]:
token_counts = Counter()

for text, label in pairs:
    token_counts.update(tokenizer.tokens(text))

vocab = set(token_counts)

print("Unique tokens:", len(vocab))
print("Total token occurrences:", sum(token_counts.values()))
print("\nMost frequent tokens:")
print(token_counts.most_common(20))

Unique tokens: 29784
Total token occurrences: 697506

Most frequent tokens:
[('.', 37371), ('i', 20475), ('the', 18563), (',', 14413), ('to', 12961), ('a', 12831), ('you', 11402), ('it', 10959), ('that', 9935), ('and', 9227), ('is', 9022), ('!', 8806), ('name', 8489), ('[', 8426), (']', 8424), ('of', 7398), ('’', 6819), ('this', 6591), ('in', 6088), ('for', 5789)]


In [14]:
# Unknown tokens map to the default vocabulary index
unk_idx = trained_emb_model.vocab.default_index

def has_embedding(token):
    return trained_emb_model.vocab.index(token) != unk_idx

# Count how many unique tokens have embeddings
covered_vocab = {
    token for token in vocab
    if has_embedding(token)
}

type_coverage = len(covered_vocab) / len(vocab)

# Frequency-weighted coverage
total_tokens = sum(token_counts.values())
covered_tokens = sum(
    count for token, count in token_counts.items()
    if has_embedding(token)
)

token_coverage = covered_tokens / total_tokens

print(f"Type coverage:  {type_coverage:.2%}")
print(f"Token coverage: {token_coverage:.2%}")

# Most frequent missing tokens
unknown = [
    (token, count)
    for token, count in token_counts.most_common()
    if not has_embedding(token)
]

print("\nMost frequent OOV tokens:")
print(unknown[:30])

Type coverage:  79.47%
Token coverage: 97.97%

Most frequent OOV tokens:
[('...', 2194), ('..', 434), ('....', 348), ('2', 336), ('3', 300), ('1', 200), ('5', 188), ('4', 130), ('10', 129), ('/s', 123), ('100', 110), ('6', 109), ('.....', 97), ('20', 95), ('😂', 75), ('30', 71), ('15', 69), ('7', 66), ('8', 65), ('downvoted', 57), ('2nd', 50), ('9', 44), ('40', 43), ('downvote', 43), ('2019', 40), ("y'all", 38), ('90', 37), ('50', 37), ('25', 37), ('12', 36)]


In [15]:
# semantic similarity between words
import numpy as np

def get_vector(word):
    idx = trained_emb_model.vocab.index(word)

    if idx == trained_emb_model.vocab.default_index:
        return None

    return np.asarray(
        trained_emb_model.vectors[idx],
        dtype=np.float32
    )


def cosine_similarity(word1, word2):
    v1 = get_vector(word1)
    v2 = get_vector(word2)

    if v1 is None or v2 is None:
        return None

    return float(
        np.dot(v1, v2) /
        (np.linalg.norm(v1) * np.linalg.norm(v2))
    )


word_pairs = [
    ("happy", "joyful"),
    ("angry", "furious"),
    ("sad", "depressed"),
    ("grateful", "thankful"),
    ("happy", "angry"),
    ("love", "hate"),
    ("good", "bad"),
]

for w1, w2 in word_pairs:
    similarity = cosine_similarity(w1, w2)

    if similarity is None:
        print(f"{w1:12} {w2:12} OOV")
    else:
        print(f"{w1:12} {w2:12} {similarity:.4f}")

happy        joyful       0.3519
angry        furious      0.3684
sad          depressed    0.7040
grateful     thankful     0.8743
happy        angry        0.4547
love         hate         0.6842
good         bad          0.7984


### Encoder (average of word embeddings)

In [16]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

tokenizer.vocab = trained_emb_model.vocab

pad_idx = trained_emb_model.vocab.index(
    trained_emb_model.pad_token
)

embedding_matrix = torch.as_tensor(
    trained_emb_model.vectors,
    dtype=torch.float32
)

encoder = part2.clslib.EmbedAndPoolEncoder(
    embedding_matrix=embedding_matrix,
    freeze_embeddings=True,
    padding_idx=pad_idx,
)

In [17]:
texts = [text for text, label in pairs[:4]]

sequences = [
    torch.tensor(tokenizer.ids(text), dtype=torch.long)
    for text in texts
]

batch = pad_sequence(
    sequences,
    batch_first=True,
    padding_value=pad_idx,
)

with torch.no_grad():
    representations = encoder(batch)

print("Input shape:", batch.shape)
print("Encoder output shape:", representations.shape)

Input shape: torch.Size([4, 22])
Encoder output shape: torch.Size([4, 200])


### Model (MLP on top of the encoder)

In [18]:
import torch
import torch.nn as nn


class MLP(nn.Module):
    """One-hidden-layer MLP, copied from part1.smlp.SparseMLP.

    Inputs: feature vectors of size `input_dim` (here: the pooled embedding, e.g. 200d GloVe).
    Process: linear into a hidden layer, ReLU + dropout, linear (num_classes) and log-softmax outputs
        log-softmax, rather than softmax, for numerical stability
    Output: logprobs over `num_classes`.

    Only change vs. part1: the first layer uses a dense matmul (x @ W) instead of
    torch.sparse.mm, since pooled embeddings are dense vectors.
    """

    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.2):
        """
        input_dim: size of the input feature space (e.g. embedding dimension)
        hidden_dim: width of the hidden layer
        num_classes: number of output labels
        dropout: fraction of hidden units zeroed during training (regularisation)
        """
        super().__init__()

        self.weight = nn.Parameter(
            torch.empty(input_dim, hidden_dim)
        )
        self.bias = nn.Parameter(
            torch.zeros(hidden_dim)
        )

        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

        self.output = nn.Sequential(
            nn.Linear(hidden_dim, num_classes),
            nn.LogSoftmax(dim=-1) # for numerical stability, it is common to use log softmax, rather than softmax
        )

        nn.init.xavier_uniform_(self.weight)

    def forward(self, x):
        """Map a batch [batch, input_dim] to class log-probabilities [batch, num_classes]."""
        x = x @ self.weight
        x = x + self.bias

        x = self.activation(x)
        x = self.dropout(x)
        # logprobs
        return self.output(x)


### Cross-validation splits

In [19]:
from part1 import experiment as p1_experiment

# Exactly the folds from Project 1 (02_experiments.ipynb): same pairs, same split function, same k and seed.
# Same folds = we can compare Project 1 and Project 2 models fold by fold (paired comparison).
folds = p1_experiment.cross_validation_split(pairs, k=config.K_FOLDS, seed=config.SEED)

for fold, split in folds.items():
    print(f"fold {fold}: {len(split['training'])} training, {len(split['validation'])} validation")

fold 0: 36356 training, 9090 validation
fold 1: 36357 training, 9089 validation
fold 2: 36357 training, 9089 validation
fold 3: 36357 training, 9089 validation
fold 4: 36357 training, 9089 validation


### Training

In [20]:
import json
from collections import OrderedDict

import numpy as np
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import part2.experiment
from src import models   # Project 1 helpers: resolve_device, count_parameters, epoch_table, summary_row

# Same settings as Project 1's one-hidden-layer MLP (02_experiments.ipynb smlp_args + part1.smlp defaults)
mlp_args = dict(hidden_dim=128, dropout=0.1)
train_args = dict(num_epochs=10, batch_size=64, learning_rate=1e-3, weight_decay=1e-4)
# Apple GPU (Metal / MPS) if available, else CUDA, else CPU. (models.resolve_device only knows CUDA.)
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
EVAL_BATCH_SIZE = 1024   # evaluation has no gradients, so bigger batches are just faster (same results)
print("device:", device)

RESULTS_DIR = PROJECT2_DIR / "results"   # same JSON format as Project 1's results/
MODELS_DIR = PROJECT2_DIR / "models"     # git-ignored (projects/*/models/)

# Tokenize every comment once; all folds reuse these ids
ids_of = {text: tokenizer.ids(text) for text, _ in tqdm(pairs, desc="tokenizing")}


def pool_encoder(freeze=True):
    """A fresh embed-and-pool encoder. Trainable embeddings get their own copy, so folds don't leak into each other."""
    matrix = embedding_matrix if freeze else embedding_matrix.clone()
    return part2.clslib.EmbedAndPoolEncoder(matrix, padding_idx=pad_idx, freeze_embeddings=freeze)


def training_one_epoch(model, dataloader, optimizer, criterion):
    """
    part2.experiment.training_one_epoch, but the running loss/accuracy stay on the device.
    Calling .item() after every batch makes the CPU wait for the GPU each step, which on MPS made training ~3x slower.
    """
    model.train()
    total_loss = torch.zeros((), device=device)
    total_correct = torch.zeros((), dtype=torch.long, device=device)
    total_examples = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logprobs = model(X_batch)
        loss = criterion(logprobs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.detach() * y_batch.size(0)
        total_correct += (logprobs.argmax(dim=1) == y_batch).sum()
        total_examples += y_batch.size(0)
    return {"loss": total_loss.item() / total_examples, "accuracy": total_correct.item() / total_examples}


@torch.no_grad()
def evaluate(model, dataloader, criterion):
    """part2.experiment.evaluate without a GPU->CPU sync per batch. Same output keys (predictions/labels as one chunk)."""
    model.eval()
    total_loss = torch.zeros((), device=device)
    preds, labels = [], []
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logprobs = model(X_batch)
        total_loss += criterion(logprobs, y_batch) * y_batch.size(0)
        preds.append(logprobs.argmax(dim=1))
        labels.append(y_batch)
    preds, labels = torch.cat(preds).cpu().numpy(), torch.cat(labels).cpu().numpy()
    return {"predictions": [preds], "labels": [labels], "loss": total_loss.item() / len(labels),
            "accuracy": float((preds == labels).mean())}


def make_loader(split, shuffle):
    dataset = part2.experiment.LabelledCorpus([ids_of[t] for t, _ in split], [int(y) for _, y in split])
    return DataLoader(dataset, batch_size=train_args["batch_size"] if shuffle else EVAL_BATCH_SIZE, shuffle=shuffle,
                      collate_fn=lambda b: part2.experiment.collate_labelled_texts(b, padding_value=pad_idx))


def train_fold(make_encoder, training_set, validation_set, save_to):
    """
    One fold, trained like Project 1 (part1.smlp): AdamW + NLLLoss for num_epochs, keep the epoch
    with the lowest validation loss and report its validation accuracy.
    Model = encoder (text ids -> one vector) followed by our MLP (vector -> log-probs).
    """
    torch.manual_seed(config.SEED)
    enc = make_encoder()
    model = nn.Sequential(enc, MLP(input_dim=enc.enc_dim, num_classes=len(outcome_space), **mlp_args)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=train_args["learning_rate"],
                                  weight_decay=train_args["weight_decay"])
    criterion = nn.NLLLoss()
    train_loader, val_loader = make_loader(training_set, True), make_loader(validation_set, False)

    trainable = {n for n, p in model.named_parameters() if p.requires_grad}
    metrics, best_epoch, best_state = [], None, None
    for epoch in range(train_args["num_epochs"]):
        train_metrics = training_one_epoch(model, train_loader, optimizer, criterion)
        val_metrics = evaluate(model, val_loader, criterion)
        metrics.append({"train_metrics": train_metrics, "val_metrics": val_metrics})
        if best_epoch is None or val_metrics["loss"] <= metrics[best_epoch]["val_metrics"]["loss"]:
            best_epoch = epoch
            # only the trained weights (the frozen 1.2M x 200 GloVe matrix would be ~1 GB per fold)
            best_state = {n: t.detach().cpu().clone() for n, t in model.state_dict().items() if n in trainable}

    save_to.parent.mkdir(parents=True, exist_ok=True)
    torch.save(best_state, f"{save_to}.pt")   # reload with model.load_state_dict(state, strict=False)
    preds = np.concatenate(metrics[best_epoch]["val_metrics"]["predictions"])
    return metrics, best_epoch, preds, models.count_parameters(model)


def cross_validate(name, make_encoder):
    """k-fold CV over Project 1's folds. Returns the same dict as Project 1's models.run_experiments."""
    out = {k: OrderedDict() for k in ("models", "predictions", "accuracies", "curves", "best_epochs")}
    for fold, split in tqdm(folds.items(), desc=name):
        save_to = MODELS_DIR / name / f"fold-{fold}" / "model"
        metrics, best_epoch, preds, n_params = train_fold(make_encoder, split["training"], split["validation"], save_to)
        out["models"][fold] = str(save_to)
        out["predictions"][fold] = preds
        out["accuracies"][fold] = metrics[best_epoch]["val_metrics"]["accuracy"]
        out["curves"][fold] = [{s: {m: ep[s][m] for m in ("loss", "accuracy")} for s in ep} for ep in metrics]
        out["best_epochs"][fold] = best_epoch
    out["n_params"] = n_params
    return out


def save_results(name, out, **meta):
    """Same JSON as Project 1's results_io.save_cv, but written to projects/project2/results/."""
    record = {
        "name": name,
        "accuracies": {str(f): float(a) for f, a in out["accuracies"].items()},
        "meta": meta,
        "n_params": out["n_params"],
        "best_epochs": {str(f): int(e) for f, e in out["best_epochs"].items()},
        "curves": {str(f): c for f, c in out["curves"].items()},
    }
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    path = RESULTS_DIR / f"{name}.json"
    path.write_text(json.dumps(record, indent=2))
    return path

tokenizing:   0%|          | 0/45446 [00:00<?, ?it/s]

In [21]:
%%time
# Embed-and-pool (frozen GloVe Twitter) + Project 1's MLP, on Project 1's folds
name = f"mlp__glove{embtable_dims}-pool"
out = cross_validate(name, lambda: pool_encoder(freeze=True))
save_results(name, out, embeddings=f"glove.twitter.27B.{embtable_dims}d", freeze_embeddings=True,
             tokenizer="BasicTokenizer(lowercase, keep punct, keep stopwords)", **mlp_args, **train_args)

acc = np.array(list(out["accuracies"].values()))
print(f"{name}: CV accuracy {acc.mean():.4f} ± {acc.std():.4f}  (mean ± std over {len(acc)} folds)")
models.epoch_table(out)

mlp__glove200-pool:   0%|          | 0/5 [00:00<?, ?it/s]

mlp__glove200-pool: CV accuracy 0.5155 ± 0.0090  (mean ± std over 5 folds)
CPU times: user 26.4 s, sys: 35 s, total: 1min 1s
Wall time: 28.8 s


,train loss,train accuracy,val loss,val accuracy
epoch,,,,
1,2.264 ± 0.004,0.404 ± 0.001,1.979 ± 0.022,0.445 ± 0.007
2,1.896 ± 0.006,0.461 ± 0.002,1.832 ± 0.022,0.477 ± 0.008
3,1.792 ± 0.006,0.483 ± 0.002,1.767 ± 0.023,0.487 ± 0.010
4,1.734 ± 0.006,0.495 ± 0.002,1.734 ± 0.025,0.493 ± 0.010
5,1.693 ± 0.005,0.504 ± 0.001,1.708 ± 0.024,0.503 ± 0.009
6,1.663 ± 0.004,0.512 ± 0.002,1.696 ± 0.020,0.501 ± 0.005
7,1.638 ± 0.004,0.517 ± 0.002,1.680 ± 0.023,0.507 ± 0.010
8,1.618 ± 0.005,0.523 ± 0.002,1.668 ± 0.021,0.510 ± 0.009
9,1.600 ± 0.005,0.526 ± 0.002,1.665 ± 0.023,0.513 ± 0.010


### Evaluation

## Experiments: embed-and-pool vs. LSTM

Every experiment is one entry in `EXPERIMENTS` (further down): a name plus what differs from `BASE`.
The loop trains each one on Project 1's 5 folds and writes
- `results/<name>.json`: fold scores, learning curves, all settings (in `meta`)
- `results/predictions/<name>.npz`: one prediction per comment, for the error analysis

`analysis.ipynb` reads both. Experiments whose results file already exists are skipped, so you can stop the loop at any time,
add entries and run it again.

Run the notebook from the top first (this section uses `pairs`, `tokenizer`, `folds`, `MLP`, `save_results`, `device`, ...).

### Smaller embedding matrix

Only the GloVe rows of words that occur in our comments (~24k instead of 1.2M). Same vectors and the same `<pad>`/`<unk>` rules as
`part2.clslib.PretrainedEmbeddings` (lowercased words, `<pad>` = zeros, `<unk>` = mean of *all* GloVe vectors), so a frozen model
gives the same results as with the full table. But trainable embeddings become feasible (~5M instead of 240M parameters) and
saved models stay small. Built once per size and cached in `data/cache/`.

In [22]:
from part2.clslib import Vocabulary
from src import evaluation

# tokens of every comment, indexed like `pairs` (same tokenizer as above)
tokens_of = [tokenizer.tokens(text) for text, _ in tqdm(pairs, desc="tokenizing")]
data_words = {t for toks in tokens_of for t in toks}
PAD, UNK = 0, 1


def small_embeddings(dims):
    """(vocab, matrix [len(vocab), dims]) for the words in our data; cached after the first call."""
    cache = CACHE_DIR / f"glove.twitter.27B.{dims}d.small.npz"
    if not cache.is_file():
        path, dims, _ = download_pretrained(dims)
        words, vectors, seen = [], [], {"<pad>", "<unk>"}
        total, n = np.zeros(dims), 0
        with open(path, encoding="utf-8") as f:
            for line in tqdm(f, desc=f"reading GloVe {dims}d", total=1_193_514):
                parts = line.split()
                word, values = parts[0].lower(), parts[1:]
                if len(values) != dims or word in seen:   # same filtering as PretrainedEmbeddings
                    continue
                seen.add(word)
                vector = np.asarray(values, dtype=np.float32)
                total, n = total + vector, n + 1
                if word in data_words:
                    words.append(word)
                    vectors.append(vector)
        unk = (total / n).astype(np.float32)
        matrix = np.vstack([np.zeros(dims, dtype=np.float32), unk, np.stack(vectors)])
        np.savez_compressed(cache, words=np.array(words), matrix=matrix)
    saved = np.load(cache)
    vocab = Vocabulary(["<pad>", "<unk>", *saved["words"].tolist()], default="<unk>", frozen=True)
    return vocab, torch.as_tensor(saved["matrix"])


_embeddings = {}


def embeddings_for(dims):
    """(matrix, ids per comment) for one GloVe size, computed once per session."""
    if dims not in _embeddings:
        vocab, matrix = small_embeddings(dims)
        _embeddings[dims] = (matrix, [[vocab.index(t) for t in toks] for toks in tokens_of])
    return _embeddings[dims]


matrix, _ = embeddings_for(200)
print("small embedding matrix:", tuple(matrix.shape))

tokenizing:   0%|          | 0/45446 [00:00<?, ?it/s]

reading GloVe 200d:   0%|          | 0/1193514 [00:00<?, ?it/s]

small embedding matrix: (23670, 200)


### LSTM encoder

The tutorial's LSTM is `part2.clslib.LSTMEncoder`: embed the words, optionally drop whole words during training (word dropout,
the recommended dropout for RNNs), run an `nn.LSTM` over the real words only (`pack_padded_sequence` skips the padding) and
return the hidden state after the last word.

`PaddedLSTMEncoder` computes the same thing, with the tutorial's suggested exercise added: a **bidirectional** option (a second LSTM
reads right to left; the encoding is both final states concatenated) and **more layers**. It does not use `pack_padded_sequence`,
which is very slow on the Mac GPU (MPS): it runs over the padded batch and reads the state at each comment's last real word.
Padding only follows the words, so it never changes that state; the right-to-left direction reads each comment reversed within its
own length. The check below confirms it matches the packed version (and the course class) up to float rounding. On MPS this
is about 3-4x faster than the packed LSTM on the CPU.

In [23]:
def reverse_within_length(h, lengths):
    """Reverse the real steps of every sequence and leave the padding in place: [a b c <pad>] -> [c b a <pad>]."""
    t = torch.arange(h.size(1), device=h.device).unsqueeze(0)    # [1, max_len]
    n = lengths.unsqueeze(1)                                      # [batch, 1]
    idx = torch.where(t < n, n - 1 - t, t)                        # [batch, max_len]
    return h.gather(1, idx.unsqueeze(-1).expand_as(h))


class PaddedLSTMEncoder(nn.Module):
    """
    part2.clslib.LSTMEncoder plus `num_layers` and `bidirectional`, without pack_padded_sequence
    (packing is very slow on Apple's MPS backend).

    Gives the same encoding as the packed version:
    - forward direction: padding only comes *after* the real words, so the output at the last real word is unaffected by it;
    - backward direction: we reverse every sequence within its own length, run a forward LSTM over that and reverse back,
      so it also never reads padding before a real word.
    One single-layer nn.LSTM per (layer, direction). With 1 layer and 1 direction the parameters are initialised exactly like
    the course class.
    """

    def __init__(self, hidden_size, embedding_matrix, padding_idx=0, freeze_embeddings=True, word_dropout=0.,
                 num_layers=1, bidirectional=False):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=freeze_embeddings,
                                                      padding_idx=padding_idx)
        self.hidden_size, self.word_dropout, self.bidirectional = hidden_size, word_dropout, bidirectional
        directions = 2 if bidirectional else 1
        self.layers = nn.ModuleList(
            nn.ModuleList(
                nn.LSTM(embedding_matrix.shape[1] if layer == 0 else hidden_size * directions, hidden_size,
                        batch_first=True)
                for _ in range(directions))
            for layer in range(num_layers))

    @property
    def enc_dim(self):
        return self.hidden_size * (2 if self.bidirectional else 1)

    def forward(self, x):
        h = self.embedding(x)                            # [batch, max_len, emb_dim]
        valid = x.ne(self.embedding.padding_idx)         # [batch, max_len], False on padding
        if self.training and self.word_dropout > 0:      # word dropout, as in part2.clslib.LSTMEncoder
            drop = (torch.rand(x.shape, device=h.device) < self.word_dropout) & valid
            h = h.masked_fill(drop.unsqueeze(-1), 0.0)
        lengths = valid.sum(-1).clamp_min(1)
        for lstms in self.layers:
            outputs = [lstms[0](h)[0]]                   # left to right
            if self.bidirectional:                       # right to left
                outputs.append(reverse_within_length(lstms[1](reverse_within_length(h, lengths))[0], lengths))
            h = torch.cat(outputs, dim=-1)               # [batch, max_len, enc_dim]
        rows = torch.arange(x.size(0), device=x.device)
        forward_state = h[rows, lengths - 1, :self.hidden_size]   # after the last word
        if not self.bidirectional:
            return forward_state
        return torch.cat([forward_state, h[:, 0, self.hidden_size:]], dim=-1)   # backward: after reading back to word 1


def make_encoder(cfg, matrix):
    """The encoder of one experiment: embed-and-pool or LSTM."""
    # nn.Embedding.from_pretrained does not copy the matrix: trainable embeddings need their own copy per fold
    matrix = matrix if cfg["freeze"] else matrix.clone()
    common = dict(embedding_matrix=matrix, padding_idx=PAD, freeze_embeddings=cfg["freeze"])
    if cfg["encoder"] == "pool":
        return clslib.EmbedAndPoolEncoder(**common)
    if cfg["encoder"] != "lstm":
        raise ValueError(f"unknown encoder {cfg['encoder']!r}")
    return PaddedLSTMEncoder(cfg["lstm_hidden"], word_dropout=cfg["word_dropout"], num_layers=cfg["lstm_layers"],
                              bidirectional=cfg["bidirectional"], **common)


# Sanity check: the LSTM encodes word order, pooling does not
_toy = {"encoder": "lstm", "freeze": True, "lstm_hidden": 4, "lstm_layers": 1, "bidirectional": False, "word_dropout": 0}
_vocab, _ = small_embeddings(200)
a, b = (torch.tensor([[_vocab.index(w) for w in s.split()]]) for s in ("not happy , sad", "not sad , happy"))
torch.manual_seed(0)
lstm_enc, pool_enc = make_encoder(_toy, matrix), make_encoder({**_toy, "encoder": "pool"}, matrix)
with torch.no_grad():
    print("pool: same encoding for both orders?", torch.allclose(pool_enc(a), pool_enc(b)))
    print("lstm: same encoding for both orders?", torch.allclose(lstm_enc(a), lstm_enc(b)))

# Sanity check: same encoding as the course's packed LSTM (same seed -> same initial weights), and for 2 layers / bidirectional
# the same as a packed nn.LSTM with the same weights
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence as _pad
_batch = _pad([torch.tensor(i) for i in embeddings_for(200)[1][:256]], batch_first=True)
torch.manual_seed(0); _course = clslib.LSTMEncoder(16, matrix).eval()
torch.manual_seed(0); _ours = PaddedLSTMEncoder(16, matrix).eval()
_ref = nn.LSTM(matrix.shape[1], 16, num_layers=2, bidirectional=True, batch_first=True)
_ours2 = PaddedLSTMEncoder(16, matrix, num_layers=2, bidirectional=True).eval()
for _l in range(2):
    for _d, _sfx in enumerate(["", "_reverse"]):
        for _p in ("weight_ih", "weight_hh", "bias_ih", "bias_hh"):
            getattr(_ours2.layers[_l][_d], f"{_p}_l0").data.copy_(getattr(_ref, f"{_p}_l{_l}{_sfx}").data)
with torch.no_grad():
    _, (_h, _) = _ref(pack_padded_sequence(_ours2.embedding(_batch), _batch.ne(PAD).sum(-1), batch_first=True,
                                           enforce_sorted=False))
    print("1 layer: max difference with course LSTMEncoder:", (_course(_batch) - _ours(_batch)).abs().max().item())
    print("2 layers, bidirectional: max difference with packed nn.LSTM:",
          (torch.cat([_h[-2], _h[-1]], -1) - _ours2(_batch)).abs().max().item())

pool: same encoding for both orders? True
lstm: same encoding for both orders? False


### Training one experiment

Same as `train_fold`/`cross_validate` above (AdamW + NLLLoss, keep the epoch with the lowest validation loss, our MLP on top of
the encoder), with three additions:
- **settings per experiment** (`cfg`) instead of the global `mlp_args`/`train_args`
- **early stopping**: stop when the validation loss has not improved for `patience` epochs (we keep the best epoch anyway, the
  rest is wasted time)
- **`train_fraction`** to train on part of each training fold (data-size experiment) and **`seed`** (variance between runs)

In [24]:
def make_loader_from(ids, idx, batch_size, shuffle):
    dataset = part2.experiment.LabelledCorpus([ids[i] for i in idx], [int(pairs[i][1]) for i in idx])
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      collate_fn=lambda b: part2.experiment.collate_labelled_texts(b, padding_value=PAD))


def run_experiment(name, cfg):
    """k-fold CV for one config. Returns the same dict as cross_validate (so save_results works)."""
    matrix, ids = embeddings_for(cfg["dims"])
    out = {k: OrderedDict() for k in ("models", "predictions", "accuracies", "curves", "best_epochs")}
    for fold, split in tqdm(folds.items(), desc=name):
        train_idx = list(split["training_ids"])
        if cfg["train_fraction"] < 1:   # same subset for every model with the same seed
            rng = np.random.default_rng(cfg["seed"] + fold)
            train_idx = rng.choice(train_idx, size=int(len(train_idx) * cfg["train_fraction"]), replace=False)
        val_idx = [int(i) for i in split["validation_ids"]]   # this order = order of the predictions

        torch.manual_seed(cfg["seed"])
        enc = make_encoder(cfg, matrix)
        model = nn.Sequential(enc, MLP(input_dim=enc.enc_dim, hidden_dim=cfg["hidden_dim"],
                                       num_classes=len(outcome_space), dropout=cfg["dropout"])).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"])
        criterion = nn.NLLLoss()
        train_loader = make_loader_from(ids, train_idx, cfg["batch_size"], shuffle=True)
        val_loader = make_loader_from(ids, val_idx, EVAL_BATCH_SIZE, shuffle=False)

        trainable = {n for n, p in model.named_parameters() if p.requires_grad}
        metrics, best_epoch, best_state = [], None, None
        for epoch in range(cfg["num_epochs"]):
            train_metrics = training_one_epoch(model, train_loader, optimizer, criterion)
            val_metrics = evaluate(model, val_loader, criterion)
            metrics.append({"train_metrics": train_metrics, "val_metrics": val_metrics})
            if best_epoch is None or val_metrics["loss"] <= metrics[best_epoch]["val_metrics"]["loss"]:
                best_epoch = epoch
                best_state = {n: t.detach().cpu().clone() for n, t in model.state_dict().items() if n in trainable}
            elif epoch - best_epoch >= cfg["patience"]:
                break   # early stopping

        save_to = MODELS_DIR / name / f"fold-{fold}" / "model"
        save_to.parent.mkdir(parents=True, exist_ok=True)
        torch.save(best_state, f"{save_to}.pt")   # reload with model.load_state_dict(state, strict=False)
        out["models"][fold] = str(save_to)
        out["predictions"][fold] = np.concatenate(metrics[best_epoch]["val_metrics"]["predictions"])
        out["accuracies"][fold] = metrics[best_epoch]["val_metrics"]["accuracy"]
        out["curves"][fold] = [{s: {m: ep[s][m] for m in ("loss", "accuracy")} for s in ep} for ep in metrics]
        out["best_epochs"][fold] = best_epoch
    out["n_params"] = models.count_parameters(model)
    return out

### The experiments

Search strategy: start from a baseline per encoder and **change one thing at a time** (not a full grid, which would be dozens of
5-fold runs). Then the data-size runs, and extra seeds for the final models to see how much results move between runs.
Edit `EXPERIMENTS` freely: once you know the best settings, point the data-size and seed runs at those.

Names follow `encoder__embedding-change` so `analysis.ipynb` can find the variants of each baseline.

In [25]:
BASE = dict(
    encoder="pool", dims=200, freeze=True,                                   # representation
    lstm_hidden=128, lstm_layers=1, bidirectional=False, word_dropout=0.0,   # LSTM only
    hidden_dim=128, dropout=0.1,                                             # MLP (same as Project 1)
    num_epochs=20, patience=3, batch_size=64, learning_rate=1e-3, weight_decay=1e-4,
    train_fraction=1.0, seed=config.SEED,
)
POOL, LSTM = dict(encoder="pool"), dict(encoder="lstm")

EXPERIMENTS = {
    # embed-and-pool: baseline, then one change at a time
    "pool__glove200": POOL,
    "pool__glove25": {**POOL, "dims": 25},
    "pool__glove200-trainable": {**POOL, "freeze": False},
    "pool__glove200-h256": {**POOL, "hidden_dim": 256},

    # LSTM: baseline, then one change at a time
    "lstm__glove200": LSTM,
    "lstm__glove200-h64": {**LSTM, "lstm_hidden": 64},
    "lstm__glove200-h256": {**LSTM, "lstm_hidden": 256},
    "lstm__glove200-wdrop0.2": {**LSTM, "word_dropout": 0.2},
    "lstm__glove200-bi": {**LSTM, "bidirectional": True},
    "lstm__glove200-2layers": {**LSTM, "lstm_layers": 2},
    "lstm__glove200-trainable": {**LSTM, "freeze": False},
    "lstm__glove25": {**LSTM, "dims": 25},

    # data size: how much does each encoder gain from more training data?
    "pool__glove200-data25": {**POOL, "train_fraction": 0.25},
    "pool__glove200-data50": {**POOL, "train_fraction": 0.5},
    "lstm__glove200-data25": {**LSTM, "train_fraction": 0.25},
    "lstm__glove200-data50": {**LSTM, "train_fraction": 0.5},

    # variance between runs: same settings, other seeds
    "pool__glove200-seed1": {**POOL, "seed": 1},
    "pool__glove200-seed2": {**POOL, "seed": 2},
    "lstm__glove200-seed1": {**LSTM, "seed": 1},
    "lstm__glove200-seed2": {**LSTM, "seed": 2},
}

In [26]:
%%time
for name, changes in EXPERIMENTS.items():
    if (RESULTS_DIR / f"{name}.json").exists():
        print(f"{name}: done already, skipped")
        continue
    cfg = {**BASE, **changes}
    out = run_experiment(name, cfg)
    save_results(name, out, **cfg, embeddings=f"glove.twitter.27B.{cfg['dims']}d",
                 tokenizer="BasicTokenizer(lowercase, keep punct, keep stopwords)")
    evaluation.save_predictions(name, out, folds)
    acc = np.array(list(out["accuracies"].values()))
    print(f"{name}: CV accuracy {acc.mean():.4f} ± {acc.std():.4f}, best epochs {list(out['best_epochs'].values())}")

pool__glove200:   0%|          | 0/5 [00:00<?, ?it/s]

pool__glove200: CV accuracy 0.5206 ± 0.0098, best epochs [17, 14, 17, 14, 14]


reading GloVe 25d:   0%|          | 0/1193514 [00:00<?, ?it/s]

pool__glove25:   0%|          | 0/5 [00:00<?, ?it/s]

pool__glove25: CV accuracy 0.4213 ± 0.0053, best epochs [18, 10, 19, 19, 13]


pool__glove200-trainable:   0%|          | 0/5 [00:00<?, ?it/s]

CPU times: user 4min 16s, sys: 4min 24s, total: 8min 41s
Wall time: 2min 40s


KeyboardInterrupt: 

In [ ]:
# What has been run so far (all comparisons and plots are in analysis.ipynb)
evaluation.sweep_table(evaluation.load_records())